In [ ]:
!pip install unsloth trl openenv-core requests matplotlib Pillow

In [ ]:
ENV_URL = "https://YOUR_USERNAME-levelforge-env.hf.space"  # placeholder
MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct"
MAX_STEPS = 200

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME, max_seq_length=1024, load_in_4bit=True,
    fast_inference=False, max_lora_rank=32, gpu_memory_utilization=0.6)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)

In [ ]:
SYSTEM_PROMPT = """You are a game level designer for a 2D platformer.
You are given an 8x16 grid and must place tiles to create a fun level.
Tiles: . (empty) # (wall) ^ (spike) $ (coin) E (enemy) P (player) G (goal)
Always wrap your thinking in <think>...</think> tags first.
Then output your edits as JSON: {"edits": [{"row": N, "col": N, "tile": "X"}], "declare_done": false}
Design for the given player personality: brave/cautious/explorer"""

In [ ]:
import requests
import json

def call_env_reward(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        try:
            action_text = completion[0]["content"]
            response = requests.post(f"{ENV_URL}/step",
                json={"reasoning": action_text[:400], "edits": [], "declare_done": False},
                timeout=10)
            data = response.json()
            rewards.append(data.get("reward", {}).get("total", 0.0))
        except:
            rewards.append(0.0)
    return rewards

def format_reward(prompts, completions, **kwargs):
    scores = []
    for c in completions:
        text = c[0]["content"]
        score = (0.5 if "<think>" in text else 0) + (0.5 if "</think>" in text else 0)
        scores.append(score)
    return scores

In [ ]:
from datasets import Dataset
import requests

scenarios_response = requests.get(f"{ENV_URL}/scenarios")
scenarios = scenarios_response.json() if scenarios_response.ok else []

# Build 200 prompts, one per scenario (repeat if needed)
data = [{"prompt": [{"role":"system","content":SYSTEM_PROMPT},
    {"role":"user","content":f"Design a {s['task_name']} level for a {s['personality']} player. Target path length: {s['target_path_len']}"}]}
    for s in (scenarios * 10)[:200]]
dataset = Dataset.from_list(data)

In [ ]:
from trl import GRPOConfig, GRPOTrainer

args = GRPOConfig(
    learning_rate=5e-6,
    optim="paged_adamw_8bit",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_generations=4,
    max_prompt_length=256,
    max_completion_length=768,
    max_steps=MAX_STEPS,
    save_steps=50,
    beta=0.04,
    loss_type="dr_grpo",
    mask_truncated_completions=True,
    report_to="none",
    output_dir="levelforge_out"
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward, call_env_reward],
    args=args,
    train_dataset=dataset
)

trainer.train()

In [ ]:
import matplotlib.pyplot as plt

rewards = [log["reward_mean"] for log in trainer.state.log_history if "reward_mean" in log]
steps = list(range(0, len(rewards) * 5, 5))

plt.figure(figsize=(10, 5))
plt.plot(steps, rewards, label="reward/mean", color="blue")
plt.xlabel("Training Step")
plt.ylabel("Reward (0-4 scale)")
plt.title("LevelForge: Reward Improvement During GRPO Training")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig("reward_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved reward_curve.png")